# Technology-area cohort review

**Status:** graduated companion notebook  
**Research thread:** cross-agency taxonomy and technology-transition reports  
**Canonical computation:** `scripts/data/build_tech_area_cohort.py`  
**Canonical verification:** `scripts/data/verify_tech_area_figures.py`

Use this notebook as the template for cohort composition, method-overlap, and contamination review. It reads generated artifacts; it does not create the canonical cohort.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
RANDOM_SEED = 20260804

In [ ]:
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID
CONFIG_PATH = REPO_ROOT / "config" / "transition_reports" / f"{AREA_ID}.yaml"
COHORT_PATH = REPORT_DIR / "cohort_keyword.csv"
OVERLAP_PATH = REPORT_DIR / "overlap_summary.json"

config = yaml.safe_load(CONFIG_PATH.read_text())
pd.DataFrame(
    [
        {"artifact": "area config", "path": CONFIG_PATH, "exists": CONFIG_PATH.exists()},
        {"artifact": "keyword cohort", "path": COHORT_PATH, "exists": COHORT_PATH.exists()},
        {"artifact": "method overlap", "path": OVERLAP_PATH, "exists": OVERLAP_PATH.exists()},
    ]
)

## Data contract

The cohort is at award grain. Review compound identity fields together (`award_id`, company, award year, and agency) because award IDs are not globally unique. The area YAML is part of the evidence: keyword rules and reviewed identity resolutions must travel with any reported count.

In [ ]:
if not COHORT_PATH.exists():
    print(
        f"Missing {COHORT_PATH.relative_to(REPO_ROOT)}. Generate it with:\n"
        f"uv run python scripts/data/build_tech_area_cohort.py --area {AREA_ID}"
    )
    cohort = pd.DataFrame()
else:
    cohort = pd.read_csv(COHORT_PATH, low_memory=False)
    print(f"{len(cohort):,} awards; {cohort['company'].nunique():,} company labels")
cohort.head()

## Composition checks

These are exploratory views. Published counts must come from the verifier, not copied notebook output.

In [ ]:
if cohort.empty:
    composition = pd.DataFrame()
else:
    composition = (
        cohort.groupby(["agency", "award_year"], dropna=False)
        .agg(awards=("award_id", "size"), firms=("company", "nunique"), amount=("award_amount", "sum"))
        .reset_index()
    )
composition.tail(20)

## Contamination and boundary sample

Inspect records near the admission boundary, especially soft-only matches and high-frequency generic terms. Preserve the deterministic sample when discussing rule changes.

In [ ]:
if cohort.empty:
    boundary_sample = pd.DataFrame()
else:
    review_columns = [
        column
        for column in ["award_id", "company", "award_year", "title", "method_a_admitted_by", "method_a_hits"]
        if column in cohort.columns
    ]
    boundary = cohort
    if "method_a_admitted_by" in cohort.columns:
        boundary = cohort[cohort["method_a_admitted_by"].ne("core")]
    boundary_sample = boundary[review_columns].sample(
        min(25, len(boundary)), random_state=RANDOM_SEED
    )
boundary_sample

## Review notes

| Observation | Proposed rule/config change | Expected precision/recall effect | Verification needed |
|---|---|---|---|
| _Record the reviewed sample_ | _None yet_ | _Unknown_ | Rebuild cohort and run `verify_tech_area_figures.py` |